# Critical Path Method (CPM)

The **Critical Path Method** finds the shortest possible duration for a project whose jobs have precedence constraints.
Each job $i$ has a processing time $p_i$ and cannot start until all its predecessors finish.

The algorithm works in two passes over the precedence DAG:

1. **Forward pass** (topological order): compute the earliest start $S_i = \max_{k \to i} C_k$ and earliest completion $C_i = S_i + p_i$ for each job.
2. **Backward pass** (reverse topological order): compute the latest completion $\bar{C}_i = \min_{i \to j} \bar{S}_j$ and latest start $\bar{S}_i = \bar{C}_i - p_i$.

A job is **critical** when $S_i = \bar{S}_i$ (equivalently $C_i = \bar{C}_i$): it has zero slack and any delay on it delays the entire project. The **critical path** is the longest path through the DAG.

In [ ]:
%config InlineBackend.figure_format = "retina"

import warnings
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import ipywidgets as widgets
from IPython.display import display, clear_output
import numpy as np
from collections import deque

warnings.filterwarnings("ignore", message=".*tight_layout.*")

plt.rcParams.update({
    "font.family": "monospace",
    "font.size": 13,
    "figure.facecolor": "#fafafa",
    "figure.dpi": 150,
    "savefig.dpi": 150,
})

COLORS = {
    "unprocessed": "#E0E0E0",
    "processed":   "#64B5F6",
    "current":     "#FFA726",
    "critical":    "#66BB6A",
    "edge":        "#BDBDBD",
    "edge_active": "#FFA726",
    "edge_critical":"#66BB6A",
    "border":      "#555555",
    "text":        "#333333",
    "bg":          "#fafafa",
}


# --------------- graph helpers ---------------

def parse_instance(durations_str, edges_str):
    """Parse text inputs into durations list and edge list."""
    durations = list(map(int, durations_str.replace(",", " ").split()))
    n = len(durations)
    edges = []
    for tok in edges_str.replace(",", " ").split():
        # accept formats like 1->2  1>2  1-2
        for sep in ["->", ">", "-"]:
            if sep in tok:
                a, b = tok.split(sep, 1)
                edges.append((int(a), int(b)))
                break
    return durations, edges


def topo_sort(n, edges):
    """Kahn's algorithm. Jobs are 1-indexed."""
    adj = {i: [] for i in range(1, n + 1)}
    indeg = {i: 0 for i in range(1, n + 1)}
    for u, v in edges:
        adj[u].append(v)
        indeg[v] += 1
    q = deque(i for i in range(1, n + 1) if indeg[i] == 0)
    order = []
    while q:
        u = q.popleft()
        order.append(u)
        for v in sorted(adj[u]):
            indeg[v] -= 1
            if indeg[v] == 0:
                q.append(v)
    return order


def compute_cpm(durations, edges):
    """Full CPM: forward pass, backward pass, critical flags."""
    n = len(durations)
    p = {i: durations[i - 1] for i in range(1, n + 1)}
    preds = {i: [] for i in range(1, n + 1)}
    succs = {i: [] for i in range(1, n + 1)}
    for u, v in edges:
        preds[v].append(u)
        succs[u].append(v)
    order = topo_sort(n, edges)

    # Forward pass
    S = {}  # earliest start
    C = {}  # earliest completion
    for i in order:
        S[i] = max((C[k] for k in preds[i]), default=0)
        C[i] = S[i] + p[i]
    C_max = max(C.values())

    # Backward pass
    S_bar = {}  # latest start
    C_bar = {}  # latest completion
    for i in reversed(order):
        C_bar[i] = min((S_bar[j] for j in succs[i]), default=C_max)
        S_bar[i] = C_bar[i] - p[i]

    critical = {i for i in range(1, n + 1) if S[i] == S_bar[i]}
    return order, p, preds, succs, S, C, S_bar, C_bar, C_max, critical


# --------------- layout ---------------

def layered_layout(n, edges):
    """Simple layered layout via longest-path layering."""
    preds = {i: [] for i in range(1, n + 1)}
    for u, v in edges:
        preds[v].append(u)
    order = topo_sort(n, edges)
    layer = {}
    for i in order:
        layer[i] = max((layer[k] + 1 for k in preds[i]), default=0)
    # group by layer
    layers = {}
    for i, l in layer.items():
        layers.setdefault(l, []).append(i)
    max_layer = max(layers.keys()) if layers else 0
    pos = {}
    for l, nodes in layers.items():
        nodes.sort()
        count = len(nodes)
        for idx, nd in enumerate(nodes):
            x = l * 2.5
            y = (idx - (count - 1) / 2) * 2.0
            pos[nd] = (x, y)
    return pos


# --------------- drawing ---------------

def draw_node(ax, x, y, label, sublabel, color, border_color=COLORS["border"], radius=0.45):
    """Draw a single node as a circle with label and optional sublabel."""
    circle = plt.Circle((x, y), radius, facecolor=color,
                         edgecolor=border_color, linewidth=2, zorder=3)
    ax.add_patch(circle)
    ax.text(x, y + (0.06 if sublabel else 0), label,
            ha="center", va="center", fontsize=13, fontweight="bold",
            color=COLORS["text"], zorder=4)
    if sublabel:
        ax.text(x, y - 0.18, sublabel,
                ha="center", va="center", fontsize=8,
                color="#666", zorder=4)


def draw_edge(ax, x1, y1, x2, y2, color=COLORS["edge"], lw=1.5, style="-"):
    """Draw a directed edge (arrow) between two node centers, stopping at radius."""
    dx, dy = x2 - x1, y2 - y1
    dist = np.hypot(dx, dy)
    if dist < 1e-6:
        return
    ux, uy = dx / dist, dy / dist
    r = 0.48
    sx, sy = x1 + ux * r, y1 + uy * r
    ex, ey = x2 - ux * r, y2 - uy * r
    ax.annotate("", xy=(ex, ey), xytext=(sx, sy),
                arrowprops=dict(arrowstyle="-|>", color=color,
                                lw=lw, linestyle=style,
                                shrinkA=0, shrinkB=0),
                zorder=2)


def draw_table(ax, headers, rows, col_widths=None, title=""):
    """Draw a simple table on an axes."""
    ax.axis("off")
    n_cols = len(headers)
    n_rows = len(rows)
    if col_widths is None:
        col_widths = [1.0 / n_cols] * n_cols
    table = ax.table(
        cellText=rows,
        colLabels=headers,
        colWidths=col_widths,
        loc="center",
        cellLoc="center",
    )
    table.auto_set_font_size(False)
    table.set_fontsize(11)
    table.scale(1, 1.5)
    # style header
    for j in range(n_cols):
        cell = table[0, j]
        cell.set_facecolor("#455A64")
        cell.set_text_props(color="white", fontweight="bold")
    # style data rows
    for i in range(1, n_rows + 1):
        for j in range(n_cols):
            cell = table[i, j]
            cell.set_facecolor("#FAFAFA" if i % 2 == 1 else "#ECEFF1")
    if title:
        ax.set_title(title, fontsize=12, fontweight="bold", pad=10)


# --------------- stepper widget ---------------

def make_stepper(step_widget, label="Step"):
    """Create a prev/next/first/last button bar tied to an IntSlider (hidden)."""
    btn_first = widgets.Button(description="|<", layout=widgets.Layout(width="40px"))
    btn_prev  = widgets.Button(description="<",  layout=widgets.Layout(width="40px"))
    btn_next  = widgets.Button(description=">",  layout=widgets.Layout(width="40px"))
    btn_last  = widgets.Button(description=">|", layout=widgets.Layout(width="40px"))
    counter   = widgets.Label(value=f"{label}: {step_widget.value}/{step_widget.max}")

    def update_label(*_):
        counter.value = f"{label}: {step_widget.value}/{step_widget.max}"

    step_widget.observe(update_label, "value")
    step_widget.observe(update_label, "max")

    def on_first(_): step_widget.value = step_widget.min
    def on_prev(_):  step_widget.value = max(step_widget.min, step_widget.value - 1)
    def on_next(_):  step_widget.value = min(step_widget.max, step_widget.value + 1)
    def on_last(_):  step_widget.value = step_widget.max

    btn_first.on_click(on_first)
    btn_prev.on_click(on_prev)
    btn_next.on_click(on_next)
    btn_last.on_click(on_last)

    return widgets.HBox([btn_first, btn_prev, counter, btn_next, btn_last])


print("Helpers loaded (retina mode).")

## Forward Pass

Process each job in **topological order**. For each job $i$:
- Earliest start: $S_i = \max_{k \in \text{pred}(i)} C_k$ (or 0 if no predecessors).
- Earliest completion: $C_i = S_i + p_i$.

Use the stepper to walk through each job being scheduled.

In [ ]:
def draw_forward_step(durations_str, edges_str, step):
    """Draw the forward-pass visualization at a given step."""
    try:
        durations, edges = parse_instance(durations_str, edges_str)
    except Exception:
        print("Invalid input. Check durations and edges format.")
        return
    n = len(durations)
    if n == 0:
        return
    order, p, preds, succs, S, C, S_bar, C_bar, C_max, critical = compute_cpm(durations, edges)
    pos = layered_layout(n, edges)

    step = min(step, n)  # 0 = initial, 1..n = after processing job order[step-1]
    processed = set(order[:step])
    current = order[step - 1] if step >= 1 else None

    # --- figure ---
    fig = plt.figure(figsize=(12, 7))
    gs = fig.add_gridspec(2, 1, height_ratios=[3, 1.5], hspace=0.35)
    ax_dag = fig.add_subplot(gs[0])
    ax_tbl = fig.add_subplot(gs[1])

    # DAG
    xs = [pos[i][0] for i in range(1, n + 1)]
    ys = [pos[i][1] for i in range(1, n + 1)]
    margin = 1.2
    ax_dag.set_xlim(min(xs) - margin, max(xs) + margin)
    ax_dag.set_ylim(min(ys) - margin, max(ys) + margin)
    ax_dag.set_aspect("equal")
    ax_dag.axis("off")

    # Draw edges
    for u, v in edges:
        x1, y1 = pos[u]
        x2, y2 = pos[v]
        if current is not None and v == current and u in processed:
            ec = COLORS["edge_active"]
            elw = 2.5
        elif u in processed and v in processed:
            ec = COLORS["processed"]
            elw = 1.8
        else:
            ec = COLORS["edge"]
            elw = 1.2
        draw_edge(ax_dag, x1, y1, x2, y2, color=ec, lw=elw)

    # Draw nodes
    for i in range(1, n + 1):
        x, y = pos[i]
        if i == current:
            nc = COLORS["current"]
        elif i in processed:
            nc = COLORS["processed"]
        else:
            nc = COLORS["unprocessed"]
        sublabel = ""
        if i in processed:
            sublabel = f"p={p[i]}"
        elif i == current:
            sublabel = f"p={p[i]}"
        draw_node(ax_dag, x, y, str(i), sublabel, nc)
        # show S/C above node if processed
        if i in processed:
            ax_dag.text(x, y + 0.65, f"S={S[i]} C={C[i]}",
                        ha="center", va="bottom", fontsize=9,
                        color=COLORS["text"], fontweight="bold",
                        bbox=dict(boxstyle="round,pad=0.15", facecolor="white",
                                  edgecolor="#ccc", alpha=0.9))

    # Status text
    if step == 0:
        status = "Initial state: no jobs processed yet."
    else:
        i = current
        pred_list = preds[i]
        if pred_list:
            pred_info = ", ".join(f"C_{k}={C[k]}" for k in pred_list)
            status = (f"Job {i}: S_{i} = max({pred_info}) = {S[i]}, "
                      f"C_{i} = {S[i]} + {p[i]} = {C[i]}")
        else:
            status = (f"Job {i}: no predecessors, S_{i} = 0, "
                      f"C_{i} = 0 + {p[i]} = {C[i]}")

    ax_dag.set_title(status, fontsize=11, fontweight="bold", pad=12,
                     color=COLORS["current"] if step > 0 else COLORS["text"])

    # Table
    headers = ["Job"] + [str(i) for i in order]
    row_p  = ["p_i"] + [str(p[i]) for i in order]
    row_s  = ["S_i"] + [str(S[i]) if i in processed else "-" for i in order]
    row_c  = ["C_i"] + [str(C[i]) if i in processed else "-" for i in order]
    draw_table(ax_tbl, headers, [row_p, row_s, row_c], title="Forward Pass")

    # highlight current column in table
    if current is not None:
        col_idx = order.index(current) + 1  # +1 for row label col
        table = ax_tbl.tables[0]
        for row_idx in range(4):  # header + 3 data rows
            cell = table[row_idx, col_idx]
            cell.set_facecolor("#FFF3E0")
            cell.set_edgecolor(COLORS["current"])

    plt.show()


# --- widgets ---
dur_input_fwd = widgets.Text(
    value="3 3 1 4 2 1 3",
    description="Durations:",
    layout=widgets.Layout(width="400px"),
)
edges_input_fwd = widgets.Text(
    value="1->2 1->3 2->4 3->5 4->6 5->6 4->7 5->7",
    description="Edges:",
    layout=widgets.Layout(width="500px"),
)
step_fwd = widgets.IntSlider(value=0, min=0, max=7, description="Step:",
                              continuous_update=True,
                              layout=widgets.Layout(width="1px", visibility="hidden"))


def _update_fwd_max(*_):
    try:
        d, e = parse_instance(dur_input_fwd.value, edges_input_fwd.value)
        step_fwd.max = len(d)
    except Exception:
        pass

dur_input_fwd.observe(_update_fwd_max, "value")
edges_input_fwd.observe(_update_fwd_max, "value")
_update_fwd_max()

out_fwd = widgets.interactive_output(
    draw_forward_step,
    {"durations_str": dur_input_fwd, "edges_str": edges_input_fwd, "step": step_fwd},
)
stepper_fwd = make_stepper(step_fwd, "Forward Step")
display(dur_input_fwd, edges_input_fwd, stepper_fwd, out_fwd)

## Backward Pass

Process each job in **reverse topological order**. For each job $i$:
- Latest completion: $\bar{C}_i = \min_{j \in \text{succ}(i)} \bar{S}_j$ (or $C_{\max}$ if no successors).
- Latest start: $\bar{S}_i = \bar{C}_i - p_i$.

The backward pass tells us how much each job can be delayed without increasing the makespan.

In [ ]:
def draw_backward_step(durations_str, edges_str, step):
    """Draw the backward-pass visualization at a given step."""
    try:
        durations, edges = parse_instance(durations_str, edges_str)
    except Exception:
        print("Invalid input. Check durations and edges format.")
        return
    n = len(durations)
    if n == 0:
        return
    order, p, preds, succs, S, C, S_bar, C_bar, C_max, critical = compute_cpm(durations, edges)
    pos = layered_layout(n, edges)
    rev_order = list(reversed(order))

    step = min(step, n)
    processed_bwd = set(rev_order[:step])
    current = rev_order[step - 1] if step >= 1 else None

    # --- figure ---
    fig = plt.figure(figsize=(12, 7))
    gs = fig.add_gridspec(2, 1, height_ratios=[3, 1.5], hspace=0.35)
    ax_dag = fig.add_subplot(gs[0])
    ax_tbl = fig.add_subplot(gs[1])

    xs = [pos[i][0] for i in range(1, n + 1)]
    ys = [pos[i][1] for i in range(1, n + 1)]
    margin = 1.2
    ax_dag.set_xlim(min(xs) - margin, max(xs) + margin)
    ax_dag.set_ylim(min(ys) - margin, max(ys) + margin)
    ax_dag.set_aspect("equal")
    ax_dag.axis("off")

    # Draw edges
    for u, v in edges:
        x1, y1 = pos[u]
        x2, y2 = pos[v]
        if current is not None and u == current and v in processed_bwd:
            ec = COLORS["edge_active"]
            elw = 2.5
        elif u in processed_bwd and v in processed_bwd:
            ec = COLORS["processed"]
            elw = 1.8
        else:
            ec = COLORS["edge"]
            elw = 1.2
        draw_edge(ax_dag, x1, y1, x2, y2, color=ec, lw=elw)

    # Draw nodes
    for i in range(1, n + 1):
        x, y = pos[i]
        if i == current:
            nc = COLORS["current"]
        elif i in processed_bwd:
            nc = COLORS["processed"]
        else:
            nc = COLORS["unprocessed"]
        sublabel = f"p={p[i]}"
        draw_node(ax_dag, x, y, str(i), sublabel, nc)
        # Show S_bar / C_bar above node if backward-processed
        if i in processed_bwd:
            ax_dag.text(x, y + 0.65,
                        f"$\\bar{{S}}$={S_bar[i]} $\\bar{{C}}$={C_bar[i]}",
                        ha="center", va="bottom", fontsize=9,
                        color=COLORS["text"], fontweight="bold",
                        bbox=dict(boxstyle="round,pad=0.15", facecolor="white",
                                  edgecolor="#ccc", alpha=0.9))
        # Show forward-pass values below node (always, as reference)
        ax_dag.text(x, y - 0.65, f"S={S[i]} C={C[i]}",
                    ha="center", va="top", fontsize=8,
                    color="#999",
                    bbox=dict(boxstyle="round,pad=0.1", facecolor="white",
                              edgecolor="#eee", alpha=0.7))

    # Status text
    if step == 0:
        status = f"Forward pass complete. C_max = {C_max}. Starting backward pass."
    else:
        i = current
        succ_list = succs[i]
        if succ_list:
            succ_info = ", ".join(f"$\\bar{{S}}$_{j}={S_bar[j]}" for j in succ_list)
            status = (f"Job {i}: $\\bar{{C}}$_{i} = min({succ_info}) = {C_bar[i]}, "
                      f"$\\bar{{S}}$_{i} = {C_bar[i]} - {p[i]} = {S_bar[i]}")
        else:
            status = (f"Job {i}: no successors, $\\bar{{C}}$_{i} = C_max = {C_max}, "
                      f"$\\bar{{S}}$_{i} = {C_max} - {p[i]} = {S_bar[i]}")

    ax_dag.set_title(status, fontsize=11, fontweight="bold", pad=12,
                     color=COLORS["current"] if step > 0 else COLORS["text"])

    # Table (show in topological order for readability)
    headers = ["Job"] + [str(i) for i in order]
    row_p    = ["p_i"]          + [str(p[i]) for i in order]
    row_s    = ["S_i"]          + [str(S[i]) for i in order]
    row_c    = ["C_i"]          + [str(C[i]) for i in order]
    row_sbar = ["S_bar_i"]      + [str(S_bar[i]) if i in processed_bwd else "-" for i in order]
    row_cbar = ["C_bar_i"]      + [str(C_bar[i]) if i in processed_bwd else "-" for i in order]
    draw_table(ax_tbl, headers, [row_p, row_s, row_c, row_sbar, row_cbar],
               title="Backward Pass")

    # highlight current column
    if current is not None:
        col_idx = order.index(current) + 1
        table = ax_tbl.tables[0]
        for row_idx in range(6):  # header + 5 rows
            cell = table[row_idx, col_idx]
            cell.set_facecolor("#FFF3E0")
            cell.set_edgecolor(COLORS["current"])

    plt.show()


# --- widgets ---
dur_input_bwd = widgets.Text(
    value="3 3 1 4 2 1 3",
    description="Durations:",
    layout=widgets.Layout(width="400px"),
)
edges_input_bwd = widgets.Text(
    value="1->2 1->3 2->4 3->5 4->6 5->6 4->7 5->7",
    description="Edges:",
    layout=widgets.Layout(width="500px"),
)
step_bwd = widgets.IntSlider(value=0, min=0, max=7, description="Step:",
                              continuous_update=True,
                              layout=widgets.Layout(width="1px", visibility="hidden"))


def _update_bwd_max(*_):
    try:
        d, e = parse_instance(dur_input_bwd.value, edges_input_bwd.value)
        step_bwd.max = len(d)
    except Exception:
        pass

dur_input_bwd.observe(_update_bwd_max, "value")
edges_input_bwd.observe(_update_bwd_max, "value")
_update_bwd_max()

out_bwd = widgets.interactive_output(
    draw_backward_step,
    {"durations_str": dur_input_bwd, "edges_str": edges_input_bwd, "step": step_bwd},
)
stepper_bwd = make_stepper(step_bwd, "Backward Step")
display(dur_input_bwd, edges_input_bwd, stepper_bwd, out_bwd)

## Critical Path

A job $i$ is **critical** when its slack is zero: $S_i = \bar{S}_i$ (equivalently, $C_i = \bar{C}_i$).
The critical path is the longest path through the DAG; its total duration equals $C_{\max}$.
Any delay on a critical job delays the entire project.

In [ ]:
def draw_critical_path(durations_str, edges_str):
    """Draw the final DAG with the critical path highlighted."""
    try:
        durations, edges = parse_instance(durations_str, edges_str)
    except Exception:
        print("Invalid input. Check durations and edges format.")
        return
    n = len(durations)
    if n == 0:
        return
    order, p, preds, succs, S, C, S_bar, C_bar, C_max, critical = compute_cpm(durations, edges)
    pos = layered_layout(n, edges)

    # --- figure ---
    fig = plt.figure(figsize=(12, 8))
    gs = fig.add_gridspec(2, 1, height_ratios=[3, 2], hspace=0.35)
    ax_dag = fig.add_subplot(gs[0])
    ax_tbl = fig.add_subplot(gs[1])

    xs = [pos[i][0] for i in range(1, n + 1)]
    ys = [pos[i][1] for i in range(1, n + 1)]
    margin = 1.2
    ax_dag.set_xlim(min(xs) - margin, max(xs) + margin)
    ax_dag.set_ylim(min(ys) - margin, max(ys) + margin)
    ax_dag.set_aspect("equal")
    ax_dag.axis("off")

    # Draw edges
    for u, v in edges:
        x1, y1 = pos[u]
        x2, y2 = pos[v]
        if u in critical and v in critical:
            # check if this is actually a critical edge (C_u == S_v)
            if C[u] == S[v]:
                ec = COLORS["edge_critical"]
                elw = 3.5
            else:
                ec = COLORS["edge"]
                elw = 1.2
        else:
            ec = COLORS["edge"]
            elw = 1.2
        draw_edge(ax_dag, x1, y1, x2, y2, color=ec, lw=elw)

    # Draw nodes
    for i in range(1, n + 1):
        x, y = pos[i]
        if i in critical:
            nc = COLORS["critical"]
            bc = "#2E7D32"
        else:
            nc = COLORS["processed"]
            bc = COLORS["border"]
        slack = S_bar[i] - S[i]
        sublabel = f"p={p[i]}"
        draw_node(ax_dag, x, y, str(i), sublabel, nc, border_color=bc)
        # Annotation above
        ax_dag.text(x, y + 0.65,
                    f"[{S[i]}, {C[i]}]",
                    ha="center", va="bottom", fontsize=9,
                    color="#2E7D32" if i in critical else COLORS["text"],
                    fontweight="bold",
                    bbox=dict(boxstyle="round,pad=0.15", facecolor="white",
                              edgecolor="#2E7D32" if i in critical else "#ccc",
                              alpha=0.9))
        # Slack below
        ax_dag.text(x, y - 0.65,
                    f"slack={slack}",
                    ha="center", va="top", fontsize=8,
                    color="#2E7D32" if slack == 0 else "#999",
                    fontweight="bold" if slack == 0 else "normal")

    # Title
    crit_str = " -> ".join(str(i) for i in order if i in critical)
    ax_dag.set_title(
        f"Critical path: {crit_str}    |    C_max = {C_max}",
        fontsize=12, fontweight="bold", pad=12, color="#2E7D32",
    )

    # Legend
    from matplotlib.lines import Line2D
    legend_elements = [
        Line2D([0], [0], marker="o", color="w", markerfacecolor=COLORS["critical"],
               markersize=12, markeredgecolor="#2E7D32", markeredgewidth=2,
               label="Critical job (slack=0)"),
        Line2D([0], [0], marker="o", color="w", markerfacecolor=COLORS["processed"],
               markersize=12, markeredgecolor=COLORS["border"], markeredgewidth=2,
               label="Non-critical job"),
        Line2D([0], [0], color=COLORS["edge_critical"], lw=3, label="Critical edge"),
        Line2D([0], [0], color=COLORS["edge"], lw=1.2, label="Non-critical edge"),
    ]
    ax_dag.legend(handles=legend_elements, loc="lower right", fontsize=9,
                  framealpha=0.9, edgecolor="#ccc")

    # Full table
    headers = ["Job"] + [str(i) for i in order]
    row_p     = ["p_i"]     + [str(p[i]) for i in order]
    row_s     = ["S_i"]     + [str(S[i]) for i in order]
    row_c     = ["C_i"]     + [str(C[i]) for i in order]
    row_sbar  = ["S_bar_i"] + [str(S_bar[i]) for i in order]
    row_cbar  = ["C_bar_i"] + [str(C_bar[i]) for i in order]
    row_slack = ["Slack"]   + [str(S_bar[i] - S[i]) for i in order]
    draw_table(ax_tbl, headers,
               [row_p, row_s, row_c, row_sbar, row_cbar, row_slack],
               title="Complete CPM Table")

    # Highlight critical columns in green
    table = ax_tbl.tables[0]
    for idx, i in enumerate(order):
        if i in critical:
            col = idx + 1
            for row_idx in range(7):  # header + 6 rows
                cell = table[row_idx, col]
                cell.set_facecolor("#E8F5E9")
                cell.set_edgecolor("#66BB6A")

    plt.show()


# --- widgets ---
dur_input_crit = widgets.Text(
    value="3 3 1 4 2 1 3",
    description="Durations:",
    layout=widgets.Layout(width="400px"),
)
edges_input_crit = widgets.Text(
    value="1->2 1->3 2->4 3->5 4->6 5->6 4->7 5->7",
    description="Edges:",
    layout=widgets.Layout(width="500px"),
)

out_crit = widgets.interactive_output(
    draw_critical_path,
    {"durations_str": dur_input_crit, "edges_str": edges_input_crit},
)
display(dur_input_crit, edges_input_crit, out_crit)